In [8]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [3]:
df=pd.read_csv('../data/football_data_cleaned.csv')
df.columns

Index(['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'MP', 'Starts',
       'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY',
       'CrdR', 'G+A-PK', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh',
       'G/SoT', 'Mn/MP', 'Min%', 'Compl', 'Subs', 'Mn/Sub', 'PPM', 'onG',
       'onGA', '+/-', '+/-90', 'On-Off', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs',
       'Int', 'TklW', 'Gls_90', 'Ast_90', 'Sh_90', 'SoT_90', 'Int_90',
       'TklW_90', 'Crs_90'],
      dtype='object')

In [4]:

df_features = [
    'Gls_90',    # Finition
    'Ast_90',    # Création
    'Sh_90',     # Volume d'attaque
    'TklW_90',   # Défense (Tacles) -> Crucial pour un profil comme Pedri
    'Int_90',    # Anticipation (Interceptions)
    'Crs_90',    # Danger provoqué
    '+/-90'      # Influence globale
]

In [5]:
df_F=df[[c for c in df_features  if c in df.columns ]]
df_F

,Gls_90,Ast_90,Sh_90,TklW_90,Int_90,Crs_90,+/-90
0,0.162162,0.202703,1.743243,0.891892,0.608108,1.459459,0.28
1,0.190880,0.000000,1.049841,1.622481,1.240721,0.858961,-0.10
2,0.172249,0.000000,0.775120,0.775120,0.861244,2.583732,0.17
3,0.146699,0.293399,0.586797,1.613692,0.660147,3.300733,1.03
4,0.043415,0.000000,0.868307,0.955137,1.128799,0.390738,-0.13
...,...,...,...,...,...,...,...
1347,0.000000,0.102215,0.562181,0.817717,1.022147,3.373083,-0.36
1348,0.041190,0.041190,0.659039,1.070938,0.988558,6.961098,-0.49
1349,0.000000,0.047418,0.284510,0.853530,1.043203,0.000000,-0.09
1350,0.158730,0.031746,0.857143,1.174603,1.238095,0.380952,0.98


In [6]:
scaler=StandardScaler()
scaler.fit(df_F)
df_scaled=scaler.transform(df_F)
df_scaled

array([[ 0.14085791,  1.10017087,  0.53559805, ..., -0.47513861,
        -0.16949219,  0.34084655],
       [ 0.31355131, -0.95799419, -0.22416831, ...,  0.90336956,
        -0.49612425, -0.18585758],
       [ 0.20151311, -0.95799419, -0.52518261, ...,  0.07646233,
         0.44203865,  0.18837956],
       ...,
       [-0.8342911 , -0.4765267 , -1.06274741, ...,  0.47296455,
        -0.96334279, -0.17199695],
       [ 0.12021983, -0.63565722, -0.4353091 , ...,  0.89764762,
        -0.75612958,  1.311091  ],
       [-0.40339325,  2.67984532,  0.27430879, ..., -0.70724391,
        -0.10586616,  1.33881227]], shape=(1352, 7))

In [9]:
model_knn=NearestNeighbors(metric='cosine',algorithm='brute')
model_knn.fit(df_scaled)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [12]:
def trouver_clones_par_poste(nom_du_joueur, n_neighbors=6):
    # 1. Trouver les infos du joueur cible
    try:
        joueur_data = df[df['Player'] == nom_du_joueur].iloc[0]
        poste_cible = joueur_data['Pos']
        query_index = df[df['Player'] == nom_du_joueur].index[0]
    except IndexError:
        return "Joueur inconnu"

    # 2. Filtrer le dataset pour ne garder que le même poste
    # On crée un masque pour les joueurs du même poste
    mask_poste = df['Pos'] == poste_cible
    df_meme_poste = df[mask_poste].copy()

    # 3. Ré-appliquer le scaler uniquement sur ce groupe (important pour la précision)
    df_F_poste = df_meme_poste[[c for c in df_features if c in df.columns]]
    df_scaled_poste = scaler.transform(df_F_poste) # On utilise le scaler déjà entraîné

    # 4. Nouveau modèle KNN sur ce groupe restreint
    model_knn_poste = NearestNeighbors(metric='cosine', algorithm='brute')
    model_knn_poste.fit(df_scaled_poste)

    # 5. Chercher le joueur dans ce nouveau groupe (son nouvel index)
    # Comme on a filtré, l'index a changé
    idx_dans_groupe = list(df_meme_poste['Player']).index(nom_du_joueur)

    distances, indices = model_knn_poste.kneighbors(
        df_scaled_poste[idx_dans_groupe].reshape(1, -1),
        n_neighbors=n_neighbors
    )

    print(f"🎯 Clones de {nom_du_joueur} au poste de {poste_cible} :\n")

    indices_flat = indices.flatten()
    dist_flat = distances.flatten()

    for i in range(1, len(indices_flat)):
        # On récupère les infos dans le dataframe filtré
        clone_row = df_meme_poste.iloc[indices_flat[i]]
        similarity = (1 - dist_flat[i]) * 100
        print(f"{i}. {clone_row['Player']} ({clone_row['Squad']}) : {similarity:.2f}%")

# Testons avec Pedri
trouver_clones_par_poste("Pedri")

🎯 Clones de Pedri au poste de MF :

1. Maxence Caqueret (Como) : 95.38%
2. Frenkie de Jong (Barcelona) : 90.53%
3. Giuliano Simeone (Atlético Madrid) : 89.17%
4. Vitinha (Paris Saint-Germain) : 89.15%
5. Santi Comesaña (Villarreal) : 87.99%
